# Parallel encoder/decoder Choi comb verification

This notebook verifies the parallel two-isometry representation against `final_comb_symbolic_original_order.pkl` using the same Choi/link-product convention as `0322originalordercomb.py`.

Important: this comparison uses the raw extracted decoder gauge (`parallel_decoder.npy`), because that is the gauge directly aligned with the saved original-order comb. The later readable decoder gauge is useful for circuit structure, but it is not the raw Choi comparison gauge used here.


In [9]:
import pickle
from pathlib import Path

import numpy as np

# Make the notebook work both from the project root and from check0402/.
ROOT = Path.cwd()
if not (ROOT / "final_comb_symbolic_original_order.pkl").exists():
    ROOT = ROOT / "check0402"
if not (ROOT / "final_comb_symbolic_original_order.pkl").exists():
    raise FileNotFoundError("Could not find final_comb_symbolic_original_order.pkl from the current working directory.")

s2, s3, s6 = np.sqrt(2), np.sqrt(3), np.sqrt(6)
np.set_printoptions(precision=6, suppress=True, linewidth=160)
print("ROOT =", ROOT.resolve())


ROOT = /home/jiayizhao/purifyu2/check2


## Step 1: write the encoder and decoder isometry matrices

Basis conventions:

- encoder rows: `|I1 I2 I3, R>`; encoder columns: `|P>`.
- decoder rows: `|F, A4>`; decoder columns: `|R, O1 O2 O3>`.

The decoder below is the raw extracted gauge matching `final_comb_symbolic_original_order.pkl`.


In [ ]:
encoder = np.array([
    [0, 0],
    [0, 0],
    [0, 0],
    [-s3/3, 0],
    [1/2, 0],
    [s3/6, 0],
    [0, 1/2],
    [0, -s3/6],
    [-1/2, 0],
    [s3/6, 0],
    [0, -1/2],
    [0, -s3/6],
    [0, 0],
    [0, s3/3],
    [0, 0],
    [0, 0],
], dtype=complex)

# Sparse matrix entries for the raw decoder.  Unlisted entries are zero.
decoder = np.zeros((32, 16), dtype=complex)


decoder_entries = [
    ( 0,  3, s2/6),
    ( 0,  5, s2/6),
    ( 0,  6, -s2/3),
    ( 0, 11, s6/6),
    ( 0, 13, -s6/6),
    ( 1,  1, -1/3),
    ( 1,  2, 1/6),
    ( 1,  4, 1/6),
    ( 1, 10, -s3/6),
    ( 1, 12, s3/6),
    ( 2,  2, -s3/6),
    ( 2,  4, s3/6),
    ( 2,  9, -s2/3 - 1/3),
    ( 2, 10, 1/6 - s2/3),
    ( 2, 12, 1/6 - s2/3),
    ( 3,  2, 1/2),
    ( 3,  4, -1/2),
    ( 3,  9, -s3/3),
    ( 3, 10, s3/6),
    ( 3, 12, s3/6),
    ( 4,  3, -1/3 + s2/6),
    ( 4,  5, -1/3 + s2/6),
    ( 4,  6, -s2/3 - 1/3),
    ( 4, 11, -s6/6),
    ( 4, 13, s6/6),
    ( 5,  0, 1),
    ( 6,  3, -s6/6),
    ( 6,  5, s6/6),
    ( 6, 11, -1/3 - s2/6),
    ( 6, 13, -1/3 - s2/6),
    ( 6, 14, -1/3 + s2/3),
    ( 7,  1, 1/3 - s2/3),
    ( 7,  2, -s2/3 - 1/6),
    ( 7,  4, -s2/3 - 1/6),
    ( 7, 10, -s3/6),
    ( 7, 12, s3/6),
    (12,  8, -1),
    (17,  3, 1/6),
    (17,  5, 1/6),
    (17,  6, -1/3),
    (17, 11, s3/6),
    (17, 13, -s3/6),
    (18,  3, s3/6),
    (18,  5, -s3/6),
    (18, 11, 1/6 - s2/3),
    (18, 13, 1/6 - s2/3),
    (18, 14, -s2/3 - 1/3),
    (19,  3, 1/2),
    (19,  5, -1/2),
    (19, 11, -s3/6),
    (19, 13, -s3/6),
    (19, 14, s3/3),
    (20,  7, -1),
    (21,  1, 1/3 + s2/3),
    (21,  2, 1/3 - s2/6),
    (21,  4, 1/3 - s2/6),
    (21, 10, -s6/6),
    (21, 12, s6/6),
    (22, 15, -1),
    (23,  3, -s2/3 - 1/6),
    (23,  5, -s2/3 - 1/6),
    (23,  6, 1/3 - s2/3),
    (23, 11, s3/6),
    (23, 13, -s3/6),
    (24,  1, -s2/3),
    (24,  2, s2/6),
    (24,  4, s2/6),
    (24, 10, -s6/6),
    (24, 12, s6/6),
    (28,  2, s6/6),
    (28,  4, -s6/6),
    (28,  9, -1/3 + s2/3),
    (28, 10, -1/3 - s2/6),
    (28, 12, -1/3 - s2/6),
]

for row, col, value in decoder_entries:
    decoder[row, col] = value

print("encoder shape:", encoder.shape)
print("decoder shape:", decoder.shape)
print("encoder isometry error:", np.linalg.norm(encoder.conj().T @ encoder - np.eye(2)))
print("decoder isometry error:", np.linalg.norm(decoder.conj().T @ decoder - np.eye(16)))


encoder shape: (16, 2)
decoder shape: (32, 16)
encoder isometry error: 1.1102230246251565e-16
decoder isometry error: 5.921795594291444e-16


## Step 2: Choi/link-product construction of the same comb matrix

The final comb has no `R` axis.  `R` is only the internal wire being linked.

There is also no `parallel` or `sequential` axis in the comb.  Both constructions must produce the same Choi matrix with the same visible wire order.

For the encoder Choi tensor we use the fully expanded axes

`P, I1, I2, I3, R, P_prime, I1_prime, I2_prime, I3_prime, R_prime`.

For the decoder Choi tensor we use the fully expanded axes

`R, O1, O2, O3, F, A4, R_prime, O1_prime, O2_prime, O3_prime, F_prime, A4_prime`.

The link product sums over both copies of the internal wire

`R` and `R_prime`.

After that, we trace out

`A4` and `A4_prime`.

The output tensor is written directly in the saved original-order comb axes

`P, I1, O1, I2, O2, I3, O3, F, P_prime, I1_prime, O1_prime, I2_prime, O2_prime, I3_prime, O3_prime, F_prime`.

So the final `256 x 256` matrix uses row/column wire order

`[P, I1, O1, I2, O2, I3, O3, F]`.


In [11]:
def matrix_to_choi(V: np.ndarray, input_dim: int, output_dim: int) -> np.ndarray:
    phi_plus = np.zeros((input_dim * input_dim, 1), dtype=complex)
    for i in range(input_dim):
        phi_plus[i * input_dim + i, 0] = 1
    phi_plus_dm = phi_plus @ phi_plus.conj().T
    i_tensor_v = np.kron(np.eye(input_dim, dtype=complex), V)
    return i_tensor_v @ phi_plus_dm @ i_tensor_v.conj().T


def parallel_choi_comb(encoder: np.ndarray, decoder: np.ndarray) -> np.ndarray:
    # c_encoder axes, in order:
    # P, I1, I2, I3, R, P_prime, I1_prime, I2_prime, I3_prime, R_prime.
    c_encoder = matrix_to_choi(encoder, input_dim=2, output_dim=16).reshape(
        2, 2, 2, 2, 2,
        2, 2, 2, 2, 2,
    )

    # c_decoder axes, in order:
    # R, O1, O2, O3, F, A4, R_prime, O1_prime, O2_prime, O3_prime, F_prime, A4_prime.
    c_decoder = matrix_to_choi(decoder, input_dim=16, output_dim=32).reshape(
        2, 2, 2, 2, 2, 16,
        2, 2, 2, 2, 2, 16,
    )

    # Link product over R and R_prime.
    # Output axes, in order:
    # P, I1, O1, I2, O2, I3, O3, F, A4,
    # P_prime, I1_prime, O1_prime, I2_prime, O2_prime, I3_prime, O3_prime, F_prime, A4_prime.
    linked_with_a4 = np.einsum(
        "pabcrqdehs,rxyzfAsuvwgB->paxbyczfAqduevhwgB",
        c_encoder,
        c_decoder,
        optimize=True,
    )

    # Partial trace over A4 and A4_prime.
    # Output axes, in order:
    # P, I1, O1, I2, O2, I3, O3, F,
    # P_prime, I1_prime, O1_prime, I2_prime, O2_prime, I3_prime, O3_prime, F_prime.
    comb_tensor = np.einsum(
        "paxbyczfAqduevhwgA->paxbyczfqduevhwg",
        linked_with_a4,
        optimize=True,
    )

    return comb_tensor.reshape(256, 256)

parallel_comb = parallel_choi_comb(encoder, decoder)
print("parallel comb shape:", parallel_comb.shape)


parallel comb shape: (256, 256)


## Step 3: compare against `final_comb_symbolic_original_order.pkl` entry by entry


In [12]:
with (ROOT / "final_comb_symbolic_original_order.pkl").open("rb") as handle:
    target_comb = np.asarray(pickle.load(handle), dtype=object)
target_comb = np.vectorize(complex)(target_comb).astype(complex)

diff = np.abs(parallel_comb - target_comb)
max_entry_error = diff.max()
fro_error = np.linalg.norm(parallel_comb - target_comb)
num_large_entries = np.sum(diff > 1e-12)

print("max_entry_error =", max_entry_error)
print("frobenius_error =", fro_error)
print("entries > 1e-12 =", num_large_entries)


max_entry_error = 5.551115123125783e-17
frobenius_error = 8.175665428932112e-16
entries > 1e-12 = 0
